# Question 1.1 — Core Contribution and Architecture of IEThresh

**Paper:** Efficiently Learning the Accuracy of Labeling Sources for Selective Sampling — Donmez, Carbonell, Schneider (KDD 2009)

**Student:** Yashi Gupta (Roll No. 230072)

## Overview

The paper addresses a practical problem in active learning: what happens when you have multiple labelers (oracles) and they aren't all equally reliable? Standard active learning assumes a single perfect oracle, but in real crowdsourcing settings you might have five annotators where two are great, two are mediocre, and one is basically guessing. IEThresh is an algorithm that simultaneously figures out *which* instances to label (the active learning part) and *which oracles to trust* (the oracle selection part). Below, I walk through the method step by step, from initialization to the full iterative loop.

## Step 1: Initialization of Oracle Reward Samples

- **Description:** Before any active learning begins, each oracle (labeler) is initialized with two dummy reward observations — one reward of 1 (success) and one reward of 0 (failure). This means every oracle starts with a mean reward of 0.5 and identical uncertainty. The confidence intervals are wide and equal for all oracles at this point.

- **Reference:** Section 3.2, paragraph describing initialization — "we smooth the confidence interval estimates by initially giving each oracle a reward of 1 and 0."

- **Purpose:** Without this initialization, oracles with no observations would have undefined confidence intervals (you can't compute a standard deviation from zero samples). The equal initialization also ensures that all oracles are explored in the first iteration, since their upper confidence bounds are identical — no oracle is unfairly favored or ignored before we have any evidence.

## Step 2: Training a Logistic Regression Classifier

- **Description:** A logistic regression model is fit on the current labeled training set T. This classifier produces posterior class probabilities P(y|x) for each unlabeled instance, which are needed for instance selection in the next step. Logistic regression is specifically chosen here because it naturally outputs calibrated probabilities, unlike something like an SVM which would need additional calibration.

- **Reference:** Section 3.2 — "we adopt a logistic regression classifier to obtain posterior class probabilities P(y|x)."

- **Purpose:** The classifier enables uncertainty-based instance selection. By having probability estimates for each class, the algorithm can identify which instances the model is most confused about. This step connects the oracle selection framework to the actual active learning objective.

## Step 3: Uncertainty-Based Instance Selection

- **Description:** From the pool of unlabeled instances, the algorithm selects the one for which the classifier is most uncertain. Formally, it picks x* = argmax_x (1 − max_y P(y|x)). For a binary problem, this is simply the instance whose predicted probability is closest to 0.5 — the decision boundary.

- **Reference:** Equation 2, Section 3.2.

- **Purpose:** Labeling the most uncertain instance gives the most information to improve the classifier's decision boundary. This is a standard active learning strategy called uncertainty sampling. The key insight is that instance selection and oracle selection are treated as separate subproblems — the paper uses a straightforward uncertainty criterion here so that the novelty can focus entirely on the oracle selection mechanism.

## Step 4: Computing Upper Confidence Intervals for Each Oracle

- **Description:** For each oracle *a*, the algorithm computes an upper confidence interval on its estimated accuracy:

  UI(a) = m(a) + t_{α/2}^{(n-1)} × s(a) / √n

  Here m(a) is the sample mean of rewards for oracle *a*, s(a) is the sample standard deviation of its rewards, n is the number of times oracle *a* has been queried, and t is the critical value from Student's t-distribution with n−1 degrees of freedom. The confidence level α is a parameter (the paper uses α = 0.05).

- **Reference:** Equation 1, Section 3.1.

- **Purpose:** The upper confidence interval captures both the estimated quality of an oracle (through the mean m(a)) AND the uncertainty about that estimate (through the s(a)/√n term). This is the same principle behind UCB-style exploration in multi-armed bandits. An oracle can have a high upper bound either because it genuinely seems good OR because we haven't queried it enough times to be sure — both are valid reasons to keep it in consideration. As n grows, the confidence interval shrinks, and the upper bound converges to the true mean.

## Step 5: Threshold-Based Oracle Selection (The Key Innovation)

- **Description:** Instead of selecting only the single best oracle (as standard Interval Estimation would), IEThresh selects ALL oracles whose upper confidence interval is within a fraction ε of the maximum upper bound. Formally, the selected set is:

  S_t = { a | UI(a) ≥ ε × max_a UI(a) }

  The parameter ε is between 0 and 1 and controls how aggressive the filtering is. When ε = 0, all oracles are selected (no filtering). When ε = 1, only oracles tied with the best upper bound are selected.

- **Reference:** Equation 4, Section 3.2.

- **Purpose:** This is what distinguishes IEThresh from plain Interval Estimation (IE). The threshold mechanism serves a dual purpose: (1) it filters out oracles whose upper bounds are clearly low, saving labeling budget, and (2) it retains multiple plausibly-good oracles for majority voting, which produces better labels than relying on a single oracle. Over time, as confidence intervals tighten with more observations, fewer oracles survive the threshold — the algorithm naturally transitions from broad exploration to focused exploitation of the best labelers.

## Step 6: Querying Selected Oracles and Computing Majority Vote

- **Description:** All oracles in the selected set S_t are asked to label the chosen instance x*. Each oracle independently provides a (possibly noisy) label. The majority vote ȳ among their responses is taken as the estimated true label. The training set is then updated: T = T ∪ {x*, ȳ}.

- **Reference:** Section 3.2, steps 6 and 7 of the algorithm outline.

- **Purpose:** Since no single labeler is guaranteed to be correct, majority vote combines opinions from the currently trusted subset of oracles. The critical difference from Repeated Labeling (the baseline) is that this vote only includes oracles that passed the threshold test — not all available oracles. As unreliable oracles get filtered out over iterations, the majority vote becomes increasingly accurate because it draws from a cleaner pool of labelers.

## Step 7: Updating Oracle Reward Estimates

- **Description:** For each oracle in the selected set S_t, a reward is computed using Equation 3:

  r̂(j) = 1 if oracle j agrees with the majority vote ȳ, and 0 otherwise.

  These binary rewards are added to each oracle's running history. Oracles that were NOT in S_t (i.e., those filtered out by the threshold) do not receive any new reward observation.

- **Reference:** Equation 3, Section 3.2.

- **Purpose:** The rewards serve as the signal for estimating oracle quality. Over many iterations, a genuinely accurate oracle will accumulate high rewards (since it tends to agree with the majority vote), while inaccurate ones will accumulate low rewards. These reward histories are exactly what feed back into Step 4 to compute updated confidence intervals. It's worth noting that using majority vote as the proxy for truth (rather than actual ground truth) is a deliberate design choice — the true labels are never known, so the algorithm bootstraps its quality estimates from the collective signal.

## Step 8: Repeat (Iterative Active Learning Loop)

- **Description:** Steps 2 through 7 are repeated for a fixed budget of iterations. With each iteration, the training set grows by one labeled instance, the classifier is retrained on the expanded dataset, the oracle confidence intervals tighten as more reward observations accumulate, and the oracle selection becomes progressively more focused on the truly reliable labelers.

- **Reference:** Section 3.2, step 9 — "Repeat 2–8."

- **Purpose:** This iterative process is the core of the active learning framework. The algorithm simultaneously learns two things in tandem: (a) which oracles to trust, and (b) a better classifier. These objectives reinforce each other — better oracle selection leads to cleaner labels, which leads to a better classifier, which leads to better uncertainty estimates for instance selection.

## Summary

IEThresh solves the problem of active learning with multiple noisy labelers of unknown reliability. The authors claim it outperforms both Repeated Labeling (asking all oracles every time) and random oracle selection because its confidence-interval-based threshold mechanism (Equation 4) efficiently identifies the most accurate oracles, reducing total labeling cost while maintaining or improving classification accuracy. The key architectural insight is combining UCB-style exploration from the multi-armed bandit literature with a soft threshold that retains multiple good oracles for majority voting, rather than committing to a single "best" labeler.